In [1]:
import os, json, math, random, re, time
from typing import Dict, List
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

from dotenv import load_dotenv
from openai import OpenAI
from openai import RateLimitError

C:\Users\laras\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------- User Settings --------------------
dataset_name    = "nvidia/HelpSteer2"
data_dir        = "preference"
split           = "train"
max_examples    = 100       
seed            = 123

openai_model    = "gpt-4o-mini" 

principles_path = "mock_principles.json"

# Outputs
out_with_pr     = f"gpt_generated_with_principles{max_examples}.json"
out_no_pr       = f"gpt_generated_no_principles{max_examples}.json"

# Repro
random.seed(seed)
load_dotenv()

key = os.getenv("OPENAI_API_KEY")
if not key:
    raise RuntimeError("Key not found")
    
client = OpenAI() 

In [ ]:
RPM_BUDGET = 450            
SPACING = 60.0 / RPM_BUDGET 

RE_MS = re.compile(r"try again in (\d+)ms", re.I)

def call_with_retry(**kwargs):
    """
    Wraps client.responses.create(**kwargs)
    - sleeps SPACING each call to honor your RPM
    - on 429, uses suggested wait from error message (if present), else exponential backoff with jitter
    """
    time.sleep(SPACING)

    backoff = 0.5       
    max_backoff = 20.0   
    max_retries = 8

    for attempt in range(max_retries):
        try:
            return client.responses.create(**kwargs)
        except RateLimitError as e:
            msg = getattr(e, "message", "") or str(e)
            m = RE_MS.search(msg)
            if m:
                wait = max(0.001, int(m.group(1)) / 1000.0)
            else:
                wait = min(max_backoff, backoff * (1.6 ** attempt)) * (0.8 + 0.4 * random.random())
            time.sleep(wait)
            continue 
        except Exception:
            raise  
    raise RuntimeError("Too many rate-limit retries; aborting.")

In [4]:

def load_principles(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list) or not all(isinstance(x, str) for x in data):
        raise ValueError("Principles file must be a JSON array of strings.")
    return data


In [5]:

def feedback_prompt(principle: str, prompt: str, respA: str, respB: str, use_principles: bool = True) -> str:
    base = (
        "Consider the following conversation between a human and an assistant.\n"
        "[HUMAN/ASSISTANT CONVERSATION]\n"
        f"[HUMAN] {prompt}\n"
        f"[ASSISTANT A] {respA}\n"
        f"[ASSISTANT B] {respB}\n"
    )
    if use_principles:
        base += (
            "[PRINCIPLE FOR MULTIPLE CHOICE EVALUATION]\n"
            f"{principle}\n"
        )
    base += (
        "Options:\n"
        "  (A) [RESPONSE A]\n"
        "  (B) [RESPONSE B]\n"
        "The answer is:"
    )
    return base


In [ ]:
_AB_RE = re.compile(r'[\s\(\[\{]*([AB])(?![A-Z0-9])', re.I)

def choose_label_openai(prompt_text: str):
    """
    Calls OpenAI Responses and returns (label, raw_text).
    label is 'A', 'B', or None if neither is detected.
    Relies on global `client` and `openai_model`.
    """
    resp = call_with_retry(
        model=openai_model,
        input=prompt_text,
        max_output_tokens=32, 
        temperature=0
    )
    try:
        txt = resp.output_text
    except Exception:
        try:
            txt = resp.output[0].content[0].text
        except Exception:
            txt = ""

    m = _AB_RE.search((txt or ""))
    label = m.group(1).upper() if m else None
    return label, (txt or "")


In [7]:

def generate_dataset(use_principles: bool, out_json: str):
    ds = load_dataset(dataset_name, data_dir=data_dir, split=split)
    n = min(max_examples, len(ds)) if 'max_examples' in globals() and max_examples is not None else len(ds)
    ds = ds.shuffle(seed=seed).select(range(n))
    print(f"[data] Subset size: {len(ds)}")

    if use_principles:
        principles = load_principles(principles_path)
        print(f"[principles] Loaded: {len(principles)}")
    else:
        principles = [""]  
        print("[principles] Skipped (use_principles=False)")

    records = []
    for row in tqdm(ds, total=len(ds), desc="Labeling"):
        prompt = row["prompt"]
        respA  = row["response_1"]
        respB  = row["response_2"]
        pr     = random.choice(principles) if use_principles else ""

        fb = feedback_prompt(
            principle=pr,
            prompt=prompt,
            respA=respA,
            respB=respB,
            use_principles=use_principles
        )

        label, judge_raw = choose_label_openai(fb)

        records.append({
            "prompt": prompt,
            "A": respA,
            "B": respB,
            "label": label,                         
            "principle": pr if use_principles else "",
            "feedback_model": openai_model,
            "judge_output": judge_raw,               
            "helpsteer_meta": {
                "preference_strength": row.get("preference_strength", None),
                "preference_statement": row.get("preference_statement", None)
            }
        })

    clean = [r for r in records if r["label"] in ("A", "B")]
    undecidable = [r for r in records if r["label"] is None]
    print(f"kept: {len(clean)} | skipped (None): {len(undecidable)}")

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(clean, f, indent=2, ensure_ascii=False)

    audit_path = out_json.replace(".json", "_undecidable.json")
    with open(audit_path, "w", encoding="utf-8") as f:
        json.dump(undecidable, f, indent=2, ensure_ascii=False)

    return {
        "saved": out_json,
        "audit": audit_path,
        "counts": {"kept": len(clean), "undecidable": len(undecidable), "total": len(records)}
    }

## Run both variants (with and without principles)

In [8]:
# With principles
generate_dataset(use_principles=True, out_json=out_with_pr)

[data] Subset size: 100
[principles] Loaded: 3


Labeling: 100%|██████████████████████████████████████████████████████████████████████| 100/100 [01:53<00:00,  1.14s/it]

kept: 100 | skipped (None): 0


{'saved': 'gpt_generated_with_principles100.json',
 'audit': 'gpt_generated_with_principles100_undecidable.json',
 'counts': {'kept': 100, 'undecidable': 0, 'total': 100}}

In [9]:
# Without principles
generate_dataset(use_principles=False, out_json=out_no_pr)

[data] Subset size: 100
[principles] Skipped (use_principles=False)


Labeling: 100%|██████████████████████████████████████████████████████████████████████| 100/100 [01:58<00:00,  1.18s/it]

kept: 100 | skipped (None): 0


{'saved': 'gpt_generated_no_principles100.json',
 'audit': 'gpt_generated_no_principles100_undecidable.json',
 'counts': {'kept': 100, 'undecidable': 0, 'total': 100}}